# Pricing and Regression Modeling
Cleaned portfolio version focused on:
- elasticity modeling
- price threshold analysis
- regression analysis
- pricing strategy insights

In [ ]:
import pandas as pd
import numpy as np
# Preprocessing Data before Analysis
# ---------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------
df = pd.read_csv("All_POS_with_Attributes.csv")

print("Raw shape:", df.shape)

# ---------------------------------------------------
# 2. RENAME KEY COLUMNS TO SIMPLE NAMES
#    (only where needed – keeps it readable & consistent)
# ---------------------------------------------------
rename_map = {
    "UPC 13 digit": "UPC",
    "Aisle Name": "Aisle",
    "Category Name": "Category",
    "Sub-Category Name": "Sub_Category",
    "Manufacturer Name": "Manufacturer",
    "Brand Franchise Name": "Brand_Franchise",
    "Brand Name": "Brand",
    "Flavor / Scent": "Flavor_Scent",
    "Type Of Meat Substituted": "Type_Of_Meat_Substituted",
    "Type Of Substitute": "Type_Of_Substitute",
    "ACV Weighted Distribution": "ACV_Weighted_Distribution"
}

df = df.rename(columns=rename_map)

# ---------------------------------------------------
# 3. PARSE TIME COLUMN → WEEK (datetime)
#    You already changed Time to look like "01-12-20"
# ---------------------------------------------------
df["Week"] = pd.to_datetime(df["Time"], format="%m-%d-%y")

# (Optional) keep Time or drop it; I like to keep both
# df = df.drop(columns=["Time"])

# ---------------------------------------------------
# 4. BASIC COLUMN NAME CLEANUP (only cosmetic)
#    - lower case
#    - replace spaces with underscores
#    - remove % and slashes
# ---------------------------------------------------
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
      .str.replace("/", "_", regex=False)
      .str.replace("%", "pct", regex=False)
      .str.lower()
)

print("Cleaned columns:")
print(df.columns.tolist())

# After this, examples:
# "Unit Sales" -> "unit_sales"
# "Dollar Sales" -> "dollar_sales"
# "ACV Weighted Distribution" -> "acv_weighted_distribution"
# "Total Ounces" -> "total_ounces"
# "Type Of Substitute" -> "type_of_substitute"

# ---------------------------------------------------
# 5. ENSURE NUMERIC TYPES FOR MEASURES
#    (Everything Circana calls sales/price/ACV)
# ---------------------------------------------------
numeric_cols = [
    # core measures
    "unit_sales", "volume_sales", "dollar_sales",
    "price_per_unit", "price_per_volume",
    "acv_weighted_distribution",
    "base_unit_sales", "base_volume_sales", "base_dollar_sales",
    "incremental_units", "incremental_volume", "incremental_dollars",
    # merch splits (only converted if present)
    "unit_sales_no_merch", "unit_sales_any_merch",
    "unit_sales_price_reductions_only", "unit_sales_feature_only",
    "unit_sales_display_only", "unit_sales_special_pack_only",
    "unit_sales_feature_and_display",
    "volume_sales_no_merch", "volume_sales_any_merch",
    "volume_sales_price_reductions_only", "volume_sales_feature_only",
    "volume_sales_display_only", "volume_sales_special_pack_only",
    "volume_sales_feature_and_display",
    "dollar_sales_no_merch", "dollar_sales_any_merch",
    "dollar_sales_price_reductions_only", "dollar_sales_feature_only",
    "dollar_sales_display_only", "dollar_sales_special_pack_only",
    "dollar_sales_feature_and_display",
    "acv_weighted_distribution_no_merch",
    "acv_weighted_distribution_any_merch",
    "acv_weighted_distribution_price_reductions_only",
    "acv_weighted_distribution_feature_only",
    "acv_weighted_distribution_display_only",
    "acv_weighted_distribution_special_pack_only",
    "acv_weighted_distribution_feature_and_display",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# For sales & volumes, NaN basically means "no sales" → set to 0
sales_like = [c for c in numeric_cols if "sales" in c or "units" in c or "volume" in c]
for col in sales_like:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Leave prices & ACV as NaN if missing (better than forcing 0)
# ---------------------------------------------------
# 6. BASIC ATTRIBUTE CLEANUP (strings)
# ---------------------------------------------------
cat_cols = [
    "upc", "product", "aisle", "category", "sub_category",
    "manufacturer", "brand_franchise", "brand",
    "package", "form", "flavor_scent", "meat_source",
    "product_type", "type_of_meat_substituted",
    "type_of_substitute", "cooked_info"
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# ---------------------------------------------------
# 7. SIMPLE TIME FEATURES FOR MODELING
# ---------------------------------------------------
df["year"] = df["week"].dt.year
df["month"] = df["week"].dt.month
df["quarter"] = df["week"].dt.quarter
df["week_number"] = df["week"].dt.isocalendar().week.astype(int)

# ---------------------------------------------------
# 8. OPTIONAL: LOG TRANSFORMS (useful for regressions)
# ---------------------------------------------------
for col in ["unit_sales", "volume_sales", "dollar_sales",
            "price_per_unit", "price_per_volume",
            "acv_weighted_distribution"]:
    if col in df.columns:
        df[f"log_{col}"] = np.log(df[col] + 1)

print("Final shape after light preprocessing:", df.shape)

In [ ]:
# ============================================
# Q1: ATTRIBUTE INTERACTION MODEL
# ============================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

# 1) Aggregate to UPC-level (across all weeks)
agg_cols = {
    "unit_sales": "sum",
    "volume_sales": "sum",
    "dollar_sales": "sum",
    "price_per_unit": "mean",
    "price_per_volume": "mean",
    "acv_weighted_distribution": "mean",
    "total_ounces": "first",
    "total_count": "first",
    "form": "first",
    "flavor_scent": "first",
    "product_type": "first",
    "type_of_substitute": "first",
    "cooked_info": "first",
    "package": "first",
    "brand": "first",
}

df_upc = (
    df.groupby("upc", as_index=False)
      .agg(agg_cols)
)

# Target variable: total unit sales per UPC (log helps handle skew)
df_upc = df_upc[df_upc["unit_sales"] > 0].copy()
df_upc["log_unit_sales"] = np.log(df_upc["unit_sales"])

# 2) Select features
feature_cols_cat = [
    "form", "flavor_scent", "product_type",
    "type_of_substitute", "cooked_info",
    "package", "brand"
]

feature_cols_num = [
    "total_ounces", "total_count",
    "price_per_unit", "price_per_volume",
    "acv_weighted_distribution"
]

X = df_upc[feature_cols_cat + feature_cols_num]
y = df_upc["log_unit_sales"]

# 3) Build preprocessing + model pipeline
cat_transformer = OneHotEncoder(handle_unknown="ignore")
preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_transformer, feature_cols_cat),
        ("num", "passthrough", feature_cols_num),
    ]
)

rf_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", rf_model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("R² on test:", r2_score(y_test, y_pred))
print("RMSE on test:", np.sqrt(mean_squared_error(y_test, y_pred)))

# 4) Feature importance (top 25 one-hot features)
# Get names of one-hot encoded cat features
ohe_feature_names = pipe.named_steps["prep"] \
                         .named_transformers_["cat"] \
                         .get_feature_names_out(feature_cols_cat)

all_feature_names = list(ohe_feature_names) + feature_cols_num

importances = pipe.named_steps["model"].feature_importances_
fi = (
    pd.DataFrame({"feature": all_feature_names, "importance": importances})
      .sort_values("importance", ascending=False)
      .head(25)
)

print(fi)

plt.figure(figsize=(8, 10))
plt.barh(fi["feature"][::-1], fi["importance"][::-1])
plt.xlabel("Importance")
plt.title("Top 25 Feature Importances – Attribute + Econ Model")
plt.tight_layout()
plt.show()

# OPTIONAL: SHAP (comment out if you don't need it)
try:
    import shap

    # Sample some rows for speed
    X_sample = X_test.sample(min(200, len(X_test)), random_state=42)

    # Transform with the same preprocessing as the model uses
    X_sample_t = pipe.named_steps["prep"].transform(X_sample)

    # If it's a sparse matrix, convert to dense
    if hasattr(X_sample_t, "toarray"):
        X_sample_t = X_sample_t.toarray()

    # Ensure numeric dtype
    X_sample_t = X_sample_t.astype(float)

    # Build explainer and compute SHAP values
    explainer = shap.TreeExplainer(pipe.named_steps["model"])
    shap_values = explainer.shap_values(X_sample_t)

    shap.summary_plot(
        shap_values,
        X_sample_t,
        feature_names=all_feature_names
    )

except ImportError:
    print("shap is not installed; skipping SHAP plots.")
except Exception as e:
    print("Could not compute SHAP values, skipping SHAP plots.")
    print("Reason:", e)

#### Q2 – Price gaps & elasticity: Are there price thresholds and gaps that change sales/velocity?
- Log-log regressions + binned price analysis.

In [ ]:
# ============================================
# Q2A: GLOBAL PRICE ELASTICITY – LOG-LOG MODEL
# ============================================
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Work at weekly UPC level, but drop rows with zero or missing values
df_price = df.copy()

df_price = df_price[
    (df_price["unit_sales"] > 0) &
    (df_price["price_per_unit"] > 0) &
    (df_price["acv_weighted_distribution"] > 0)
].copy()

df_price["ln_unit_sales"] = np.log(df_price["unit_sales"])
df_price["ln_price_unit"] = np.log(df_price["price_per_unit"])
df_price["ln_acv"] = np.log(df_price["acv_weighted_distribution"])

# Simple elasticity model (you can add brand dummies later if you want)
m_unit = smf.ols(
    "ln_unit_sales ~ ln_price_unit + ln_acv",
    data=df_price
).fit()

print(m_unit.summary())
print("\nPrice elasticity (unit):", m_unit.params["ln_price_unit"])
print("ACV elasticity (unit):", m_unit.params["ln_acv"])

In [ ]:
df_price_vol = df[
    (df["volume_sales"] > 0) &
    (df["price_per_volume"] > 0) &
    (df["acv_weighted_distribution"] > 0)
].copy()

df_price_vol["ln_volume_sales"] = np.log(df_price_vol["volume_sales"])
df_price_vol["ln_price_vol"] = np.log(df_price_vol["price_per_volume"])
df_price_vol["ln_acv"] = np.log(df_price_vol["acv_weighted_distribution"])

m_vol = smf.ols(
    "ln_volume_sales ~ ln_price_vol + ln_acv",
    data=df_price_vol
).fit()

print(m_vol.summary())
print("\nPrice elasticity (volume):", m_vol.params["ln_price_vol"])
print("ACV elasticity (volume):", m_vol.params["ln_acv"])

In [ ]:
# ============================================
# Q2B: ELASTICITY BY BRAND
# ============================================
elasticity_by_brand = []

for brand, sub in df_price.groupby("brand"):
    if len(sub) < 100:  # skip tiny brands
        continue
    try:
        model_b = smf.ols(
            "ln_unit_sales ~ ln_price_unit + ln_acv",
            data=sub
        ).fit()
        elasticity_by_brand.append({
            "brand": brand,
            "price_elasticity_unit": model_b.params.get("ln_price_unit", np.nan),
            "acv_elasticity_unit": model_b.params.get("ln_acv", np.nan),
            "n_obs": len(sub)
        })
    except Exception as e:
        print("Error for brand", brand, ":", e)

elasticity_brand_df = pd.DataFrame(elasticity_by_brand)
elasticity_brand_df = elasticity_brand_df.sort_values("price_elasticity_unit")
print(elasticity_brand_df)

In [ ]:
# ============================================
# Q2C: PRICE BANDS / THRESHOLDS
# ============================================
import matplotlib.pyplot as plt

# Use only reasonable price range to avoid super outliers
df_bins = df[
    (df["price_per_unit"] > 0) &
    (df["price_per_unit"] < df["price_per_unit"].quantile(0.99))
].copy()

# Create price bands (e.g., 10 equal-width bins)
df_bins["price_band"] = pd.qcut(df_bins["price_per_unit"], q=10)

band_summary = (
    df_bins.groupby("price_band")
           .agg(
               avg_price=("price_per_unit", "mean"),
               avg_unit_sales=("unit_sales", "mean"),
               avg_volume_sales=("volume_sales", "mean")
           )
           .reset_index()
)

print(band_summary)

plt.figure(figsize=(6,4))
plt.plot(band_summary["avg_price"], band_summary["avg_unit_sales"], marker="o")
plt.xlabel("Average Price per Unit (by band)")
plt.ylabel("Average Unit Sales")
plt.title("Price Bands vs Avg Unit Sales")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Q2-A. Price Bands vs Average Unit Sales
plt.figure(figsize=(8,4))
plt.plot(band_summary["avg_price"], band_summary["avg_unit_sales"], marker="o")
plt.xlabel("Average Price per Unit (Price Band)")
plt.ylabel("Average Unit Sales")
plt.title("Price Bands vs Average Unit Sales")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Q2-B. Price vs Unit Velocity per ACV (Category)
plt.figure(figsize=(6,5))
plt.scatter(
    upc_perf["price_per_unit"],
    upc_perf["unit_velocity_per_acv"],
    s=20,
    alpha=0.6
)
plt.xlabel("Price per Unit")
plt.ylabel("Unit Velocity per ACV")
plt.title("Price vs Velocity (per UPC)")
plt.ylim(0, 5000)   # adjust as needed
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Q2-C. Price vs Velocity – Gardein Highlighted
is_gardein = upc_perf["brand"].str.upper().str.contains("GARDEIN")

plt.figure(figsize=(6,5))
plt.scatter(
    upc_perf.loc[~is_gardein, "price_per_unit"],
    upc_perf.loc[~is_gardein, "unit_velocity_per_acv"],
    s=15, alpha=0.4, label="Other Brands"
)
plt.scatter(
    upc_perf.loc[is_gardein, "price_per_unit"],
    upc_perf.loc[is_gardein, "unit_velocity_per_acv"],
    s=30, alpha=0.9, label="Gardein"
)
plt.xlabel("Price per Unit")
plt.ylabel("Unit Velocity per ACV")
plt.title("Gardein vs Category: Price vs Velocity")
plt.legend()
plt.ylim(0, 5000)   # adjust as needed
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Q2-D. Brand-Level Elasticity Comparison Table (Pretty Print)
top_elastic = elasticity_brand_df.sort_values("price_elasticity_unit").head(15)

plt.figure(figsize=(8,6))
plt.barh(top_elastic["brand"], top_elastic["price_elasticity_unit"])
plt.xlabel("Price Elasticity (Unit)")
plt.title("Most Price-Sensitive Brands")
plt.tight_layout()
plt.show()

In [ ]:
# Q2-E. ACV Elasticity Comparison Chart
top_acv = elasticity_brand_df.sort_values("acv_elasticity_unit", ascending=False).head(15)

plt.figure(figsize=(8,6))
plt.barh(top_acv["brand"], top_acv["acv_elasticity_unit"])
plt.xlabel("ACV Elasticity")
plt.title("Brands Most Sensitive to Distribution Levels")
plt.tight_layout()
plt.show()